In [33]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

## State Backend

In [34]:
import os
from deepagents import create_deep_agent
from deepagents.backends import StateBackend
agent=create_deep_agent(model="ollama:qwen3:4b")
agent2=create_deep_agent(model="openai:gpt-5.4",backend=StateBackend())

result = agent.invoke({"messages":[{"role":"user","content":(
 "Create a file  at /notes/todo.txt with exactly this content:\n"
 "1. Record video \n2.edit video \n3.Upload video\n"
 "Then tell me you have done it")}]})

print("\n------Agent Reply------")
print(result["messages"][-1].content)




------Agent Reply------
The file `/notes/todo.txt` has been created with the exact content you specified. ✅


### File system backend(local disk)

In [40]:
from deepagents.backends import FilesystemBackend
ROOT = "."
agent = create_deep_agent(model="ollama:qwen3:4b",backend=FilesystemBackend(root_dir=ROOT,virtual_mode=True))
result = agent.invoke({"messages":[{"role":"user","content":(
 "Create a file  at /notes/todo.txt with exactly this content:\n"
 "1. Record video \n2.edit video \n3.Upload video\n"
 "Then tell me you have done it")}]})



### Deep agent store backend verification
 Create a deepagent backend by LanGraph store, invoke it to write a file on one thread, then proves the backends works by reading that file back on diffrent thread, something StateBackend can not do



In [41]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend
store = InMemoryStore()

agent = create_deep_agent(
    model="openai:gpt-5.4",
    backend=StoreBackend(
        # Local dev: static namespace. No deployment runtime needed.
        # In a LangSmith Deployment you'd use the user-identity version.
        namespace=lambda rt: ("demo-user",),
    ),
    store=store,
)
print("✅ Agent created with StoreBackend + static namespace.")


✅ Agent created with StoreBackend + static namespace.


In [42]:
import os
import uuid
# -----------------------------------------------------------------------------
# 2. THREAD 1 — write a file
# -----------------------------------------------------------------------------
thread_1 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Create a file at /notes/todo.txt with exactly this content:\n"
                "1. Record video\n2. Edit video\n3. Upload video\n"
                "Then tell me you've done it."
            )
        }]
    },
    config=thread_1,
)

print("\n--- Agent reply (thread 1) --------------------------------------")
print(result["messages"][-1].content)


--- Agent reply (thread 1) --------------------------------------
[{'type': 'text', 'text': 'Done.', 'annotations': [], 'id': 'msg_069115ebc2619763006aaec1f784b487d18264908953c365db', 'phase': 'final_answer'}]


In [43]:
# --- Thread 2: read back on a DIFFERENT thread ---------------------------
thread_2 = {"configurable": {"thread_id": str(uuid.uuid4())}}
followup = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me verbatim."
    }]},
    config=thread_2,
)
print("\n--- Read-back on a different thread ---")
print(followup["messages"][-1].content)


--- Read-back on a different thread ---
[{'type': 'text', 'text': '1. Record video\n2. Edit video\n3. Upload video', 'annotations': [], 'id': 'msg_03028f09d4af2344006aaec24542c087d1aa44390789eb68a2', 'phase': 'final_answer'}]
